# Assemble All Relevance Scores

| Item | Description |
|---|---|
| Goal | Put the sentence-based scores (T3.8.1) and the summary-based scores (T3.8.2) into one firm-level table, long and wide, and show how the measures relate across firms. |
| Input | `output/T3.8.1/score_sentences.csv`; `output/T3.8.2/score_summaries.csv`; `output/T3.4.1/filings.csv`. |
| Output | `output/T3.8.3/firm_relevance_scores.csv`: one row per firm, sample, and measure (long). `output/T3.8.3/firm_relevance_scores_wide.csv`: one row per firm and sample, one column per measure. `output/T3.8.3/measure_correlations.csv`: Spearman correlations among the measures on the three-year sample. The second cell of this notebook describes every measure. |
| Unit | A firm (gvkey) within a sample. |
| Measures | From the sentences, per topic vector (definition, centroid): `binary`, `share`; samples all years and three years. From the summaries, per topic-side vector (definition, centroid, policy summary): `firm_summary`, `yearly_mean`; both samples, prompt version 1.3. Ten measures in all on each sample; the all-years sample is the main one. The mean cosine of the matched sentences was dropped on 2026-10-25 because it varies only within the narrow band above the threshold and tracks nothing. |
| Note | Industry codes are not attached here; the industry grouping is still an open item in the plan. |

See `README.md` in the TEMP folder for the surrounding steps.


## What each measure is

Everything needed to read the table, in one place. Cosine similarity is always between two unit-length embeddings from `BAAI/bge-large-en-v1.5` (CLS pooling), so it is the dot product. The thresholds are the ones fixed by hand in T3.6.5.

**Topic side** (three vectors, all for Topic 68, consumer data privacy legislation):

- **definition vector**: the embedding of the codebook label and definition (T3.5.1): *"consumer data privacy legislation: This topic encompasses federal and state laws, proposed bills, and regulatory frameworks specifically addressing the collection, use, and protection of consumer personal information."* Sentence threshold 0.67.
- **centroid vector**: the unit-length mean of the embeddings of the 374 hearing statements coded to Topic 68, support and concern together (T3.5.3). Sentence threshold 0.75.
- **policy summary vector**: the embedding of the 221-word summary of what those statements say the legislation does, written by Qwen3.5-4B by map over hearing years and one reduce (T3.5.5, T3.5.7):
  > The legislation imposes rigorous obligations on tech giants, broadband providers, healthcare organizations, drone operators, and financial institutions to secure consumer data and prevent its unauthorized sale or misuse. It grants individuals specific rights, including the ability to opt in or out of data collection, demand clear and conspicuous privacy policies, control their digital communications, and establish data "firewalls." Companies are required to implement "privacy by design," provide breach notifications within 72 hours, and ensure that sensitive information such as health records, social security numbers, and geolocation data is protected from discrimination and identity theft. The regulatory scope targets a wide array of data practices, ranging from the deceptive monitoring of online activities and the unauthorized mapping of consumer movements to the commercialization of private data and the use of algorithms to amplify or suppress content. Specific actors include research centers, third-party aggregators, and federal agencies, with particular emphasis on preventing the export of products and the sharing of personal information without consent. The framework addresses diverse data types, including directory information, voice recordings, and data gathered by connected vehicles, aiming to balance national security interests with civil liberties. Ultimately, the legislation seeks to replace fragmented standards with a uniform national baseline that mandates transparency, enforces accountability, and empowers consumers to make informed choices regarding their personal information in the digital economy.

**Firm side** (three kinds of text):

- **Item 1 sentences**: every sentence of the firm's 10-K Item 1 sections in the sample, split with pysbd and embedded one by one (T3.6.1, T3.6.3). Samples: `three_years` = fiscal 2010, 2015, 2020; `all_years` = every fiscal year 1999 to 2025.
- **yearly summary**: one paragraph per filing, written by Qwen3.5-4B from the Item 1 text by map and reduce (T3.7.2; prompt version `item1-mapreduce-1.3`, lines-of-business family, 150 to 200 words asked), embedded whole (T3.7.6). Both samples.
- **firm summary**: one paragraph per firm, written by Qwen3.5-4B from the firm's yearly summaries (T3.7.4; same version, 200 to 250 words asked) over the firm's years in the sample, embedded whole (T3.7.6). One per sample.

**Measures** (`measure_id` = method | topic vector | measure):

- **sentences | definition | binary**. Topic side: definition vector. Firm side: the firm's Item 1 sentences in the sample. How to measure: cosine of every sentence with the definition vector; the firm scores 1 if at least one sentence is at or above 0.67, otherwise 0.
- **sentences | definition | share**. Topic side: definition vector. Firm side: the firm's Item 1 sentences in the sample. How to measure: the number of sentences at or above 0.67 divided by all of the firm's Item 1 sentences in the sample.
- **sentences | centroid | binary**. Topic side: centroid vector. Firm side: the firm's Item 1 sentences in the sample. How to measure: cosine of every sentence with the centroid vector; the firm scores 1 if at least one sentence is at or above 0.75, otherwise 0.
- **sentences | centroid | share**. Topic side: centroid vector. Firm side: the firm's Item 1 sentences in the sample. How to measure: the number of sentences at or above 0.75 divided by all of the firm's Item 1 sentences in the sample.
- **summaries | definition | firm_summary**. Topic side: definition vector. Firm side: the firm summary. How to measure: the cosine of the firm summary embedding with the definition vector.
- **summaries | definition | yearly_mean**. Topic side: definition vector. Firm side: the firm's yearly summaries. How to measure: the cosine of each yearly summary embedding with the definition vector, averaged over the firm's fiscal years in the sample.
- **summaries | centroid | firm_summary**. Topic side: centroid vector. Firm side: the firm summary. How to measure: the cosine of the firm summary embedding with the centroid vector.
- **summaries | centroid | yearly_mean**. Topic side: centroid vector. Firm side: the firm's yearly summaries. How to measure: the cosine of each yearly summary embedding with the centroid vector, averaged over the firm's fiscal years in the sample.
- **summaries | policy_summary | firm_summary**. Topic side: policy summary vector. Firm side: the firm summary. How to measure: the cosine of the firm summary embedding with the policy summary vector.
- **summaries | policy_summary | yearly_mean**. Topic side: policy summary vector. Firm side: the firm's yearly summaries. How to measure: the cosine of each yearly summary embedding with the policy summary vector, averaged over the firm's fiscal years in the sample.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.8.3"  # the identifier names this notebook's output folder
TEMP_OUTPUT = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output"
OUTPUT_FOLDER = TEMP_OUTPUT / NOTEBOOK_NAME
SENTENCE_SCORES_FILE = TEMP_OUTPUT / "T3.8.1/score_sentences.csv"
SUMMARY_SCORES_FILE = TEMP_OUTPUT / "T3.8.2/score_summaries.csv"
FILINGS_FILE = TEMP_OUTPUT / "T3.4.1/filings.csv"

CORRELATION_SAMPLE = "all_years"  # the main sample
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.3


## Step 3. Load the two score tables

Each already is long; the summary table's `level` plays the role of the sentence table's `measure`, so both go into one `measure` column. A `measure_id` names the measure in full: method, topic vector, measure.


In [3]:
sentence_scores = pd.read_csv(SENTENCE_SCORES_FILE, dtype={"gvkey": str})
summary_scores = pd.read_csv(SUMMARY_SCORES_FILE, dtype={"gvkey": str})
summary_scores = summary_scores.rename(columns={"level": "measure"})
columns = ["gvkey", "conm", "method", "sample", "topic_vector", "measure", "n_filings", "score"]
scores = pd.concat([
    sentence_scores[columns + ["threshold", "n_sentences", "n_above_threshold"]],
    summary_scores[columns + ["family", "prompt_version"]],
], ignore_index=True)
scores["measure_id"] = scores["method"] + " | " + scores["topic_vector"] + " | " + scores["measure"]
assert not scores.duplicated(["gvkey", "sample", "measure_id"]).any()
print(f"Sentence rows: {len(sentence_scores):,}; summary rows: {len(summary_scores):,}; combined: {len(scores):,}")
print(f"Measures: {scores['measure_id'].nunique()}")


Sentence rows: 7,596; summary rows: 11,394; combined: 18,990
Measures: 10


## Step 4. Check the firms against the sample list

Every firm of each sample appears once per measure.


In [4]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
firms_in = {"all_years": set(filings.loc[filings["in_all_years"].eq(1), "gvkey"]), "three_years": set(filings.loc[filings["in_three_years"].eq(1), "gvkey"])}
coverage = scores.groupby(["sample", "measure_id"]).agg(firms=("gvkey", "nunique"), firms_with_score=("score", lambda s: int(s.notna().sum())))
for (sample, measure_id), row in coverage.iterrows():
    assert row["firms"] == len(firms_in[sample]), (sample, measure_id)
display(coverage)


firms  firms_with_score
sample      measure_id                                                        
all_years   sentences | centroid | binary               1058              1058
            sentences | centroid | share                1058              1058
            sentences | definition | binary             1058              1058
            sentences | definition | share              1058              1058
            summaries | centroid | firm_summary         1058              1058
            summaries | centroid | yearly_mean          1058              1058
            summaries | definition | firm_summary       1058              1058
            summaries | definition | yearly_mean        1058              1058
            summaries | policy_summary | firm_summary   1058              1058
            summaries | policy_summary | yearly_mean    1058              1058
three_years sentences | centroid | binary                841               841
            sentences | centroid | share                 841               841
            sentences | definition | binary              841               841
            sentences | definition | share               841               841
            summaries | centroid | firm_summary          841               841
            summaries | centroid | yearly_mean           841               841
            summaries | definition | firm_summary        841               841
            summaries | definition | yearly_mean         841               841
            summaries | policy_summary | firm_summary    841               841
            summaries | policy_summary | yearly_mean     841               841

## Step 5. Build the wide table

One row per firm and sample, one column per measure.


In [5]:
wide = scores.pivot_table(index=["gvkey", "conm", "sample"], columns="measure_id", values="score", aggfunc="first").reset_index()
wide.columns.name = None
n_filings = scores.drop_duplicates(["gvkey", "sample"]).set_index(["gvkey", "sample"])["n_filings"]
wide["n_filings"] = [n_filings.loc[(g, s)] for g, s in zip(wide["gvkey"], wide["sample"])]
measure_columns = sorted(c for c in wide.columns if " | " in c)
wide = wide[["gvkey", "conm", "sample", "n_filings"] + measure_columns]
print(f"Wide table: {len(wide):,} rows (firm x sample) and {len(measure_columns)} measure columns")
display(wide.loc[wide["sample"].eq(CORRELATION_SAMPLE)].head(3).T)


Wide table: 1,899 rows (firm x sample) and 10 measure columns


,0,2,4
gvkey,001013,001045,001075
conm,ADC TELECOMMUNICATIONS INC,AMERICAN AIRLINES GROUP INC,PINNACLE WEST CAPITAL CORP
sample,all_years,all_years,all_years
n_filings,12,22,15
sentences | centroid | binary,0.0,1.0,0.0
sentences | centroid | share,0.0,0.000412,0.0
sentences | definition | binary,0.0,0.0,0.0
sentences | definition | share,0.0,0.0,0.0
summaries | centroid | firm_summary,0.519826,0.498365,0.47419
summaries | centroid | yearly_mean,0.48966,0.500428,0.493738


## Step 6. Correlations among the measures on the all-years sample

Spearman rank correlations across firms on the all-years sample. The binary measure is 0 or 1, so its correlations are coarse.


In [6]:
main_sample = wide.loc[wide["sample"].eq(CORRELATION_SAMPLE), measure_columns]
correlations = main_sample.corr(method="spearman")
pd.set_option("display.width", 250)
display(correlations.round(2))
share_rows = [c for c in measure_columns if c.endswith("| share")]
print("The sentence share against every other measure:")
display(correlations.loc[share_rows].drop(columns=share_rows).round(2).T)


,sentences | centroid | binary,sentences | centroid | share,sentences | definition | binary,sentences | definition | share,summaries | centroid | firm_summary,summaries | centroid | yearly_mean,summaries | definition | firm_summary,summaries | definition | yearly_mean,summaries | policy_summary | firm_summary,summaries | policy_summary | yearly_mean
sentences | centroid | binary,1.00,0.99,0.77,0.79,0.20,0.24,0.10,0.17,0.19,0.22
sentences | centroid | share,0.99,1.00,0.77,0.81,0.21,0.25,0.12,0.19,0.20,0.24
sentences | definition | binary,0.77,0.77,1.00,0.98,0.20,0.24,0.14,0.20,0.18,0.21
sentences | definition | share,0.79,0.81,0.98,1.00,0.22,0.26,0.16,0.22,0.20,0.22
summaries | centroid | firm_summary,0.20,0.21,0.20,0.22,1.00,0.79,0.73,0.60,0.79,0.68
summaries | centroid | yearly_mean,0.24,0.25,0.24,0.26,0.79,1.00,0.63,0.77,0.66,0.80
summaries | definition | firm_summary,0.10,0.12,0.14,0.16,0.73,0.63,1.00,0.83,0.65,0.56
summaries | definition | yearly_mean,0.17,0.19,0.20,0.22,0.60,0.77,0.83,1.00,0.54,0.66
summaries | policy_summary | firm_summary,0.19,0.20,0.18,0.20,0.79,0.66,0.65,0.54,1.00,0.87
summaries | policy_summary | yearly_mean,0.22,0.24,0.21,0.22,0.68,0.80,0.56,0.66,0.87,1.00


The sentence share against every other measure:


,sentences | centroid | share,sentences | definition | share
sentences | centroid | binary,0.99,0.79
sentences | definition | binary,0.77,0.98
summaries | centroid | firm_summary,0.21,0.22
summaries | centroid | yearly_mean,0.25,0.26
summaries | definition | firm_summary,0.12,0.16
summaries | definition | yearly_mean,0.19,0.22
summaries | policy_summary | firm_summary,0.20,0.20
summaries | policy_summary | yearly_mean,0.24,0.22


## Step 7. The top ten firms under each measure on the all-years sample

Names only, side by side, so the reader sees where the measures agree.


In [7]:
top = {}
for column in measure_columns:
    ordered = wide.loc[wide["sample"].eq(CORRELATION_SAMPLE), ["conm", column]].dropna().sort_values(column, ascending=False)
    if column.endswith("| binary"):
        continue  # a 0/1 measure has no meaningful top ten
    top[column.replace("sentences | ", "sent | ").replace("summaries | ", "summ | ").replace("policy_summary", "policy")] = ordered["conm"].head(10).str.slice(0, 26).tolist()
display(pd.DataFrame(top, index=range(1, 11)))


,sent | centroid | share,sent | definition | share,summ | centroid | firm_summary,summ | centroid | yearly_mean,summ | definition | firm_summary,summ | definition | yearly_mean,summ | policy | firm_summary,summ | policy | yearly_mean
1,META PLATFORMS INC,EQUIFAX INC,CHARTER ONE FINANCIAL INC,CHARTER ONE FINANCIAL INC,WACHOVIA CORP-OLD,CHARTER ONE FINANCIAL INC,VISA INC,GEN DIGITAL INC
2,VERALTO CORP,TRADE DESK INC,VISA INC,GEN DIGITAL INC,CHARTER ONE FINANCIAL INC,ALLSTATE CORP,PALANTIR TECHNOLOG INC,VERIZON COMMUNICATIONS INC
3,DUN & BRADSTREET HOLDNGS I,APPLOVIN CORP,GEN DIGITAL INC,ENTERGY CORP,ALLSTATE CORP,WACHOVIA CORP-OLD,GEN DIGITAL INC,CHARTER ONE FINANCIAL INC
4,TWITTER INC,WESTERN UNION CO,ALLSTATE CORP,PROVIDIAN FINANCIAL CORP,CITIZENS FINANCIAL GROUP I,CITIZENS FINANCIAL GROUP I,COMVERSE TECHNOLOGY INC,EQUIFAX INC
5,TRADE DESK INC,META PLATFORMS INC,PALANTIR TECHNOLOG INC,WARNER-LAMBERT CO,VISA INC,VERIZON COMMUNICATIONS INC,NIELSEN HOLDINGS PLC,CHARTER COMMUNICATIONS INC
6,BROADRIDGE FINANCIAL SOLUT,TWITTER INC,ENTERGY CORP,WACHOVIA CORP-OLD,MICROSOFT CORP,EQUIFAX INC,DELL TECHNOLOGIES INC,MOTOROLA MOBILITY HLDGS IN
7,AMERICAN GREETINGS -CL A,PAYPAL HOLDINGS INC,ADT CORP,EQUIFAX INC,NIELSEN HOLDINGS PLC,CONSUMERS ENERGY CO,CHARTER ONE FINANCIAL INC,SPRINT CORP
8,PAYPAL HOLDINGS INC,DUN & BRADSTREET HOLDNGS I,WARNER-LAMBERT CO,ALLSTATE CORP,NATIONAL CITY CORP,PACTIV CORP,ALPHABET INC,VISA INC
9,BREAD FINANCIAL HOLDINGS I,OPTIMUM COMMUNICATIONS INC,PALO ALTO NETWORKS INC,MCAFEE INC,PROVIDIAN FINANCIAL CORP,SYNCHRONY FINANCIAL,VERIZON COMMUNICATIONS INC,BROADCOM INC
10,FORTIVE CORP,BREAD FINANCIAL HOLDINGS I,WACHOVIA CORP-OLD,BEST BUY CO INC,EQUIFAX INC,NATIONAL CITY CORP,MOTOROLA MOBILITY HLDGS IN,PALO ALTO NETWORKS INC


## Step 8. Save the long table, the wide table, and the correlations


In [8]:
long_file = OUTPUT_FOLDER / "firm_relevance_scores.csv"
scores_out = scores[["gvkey", "conm", "sample", "method", "topic_vector", "measure", "measure_id", "n_filings", "n_sentences", "n_above_threshold", "threshold", "family", "prompt_version", "score"]]
scores_out.sort_values(["sample", "measure_id", "gvkey"]).to_csv(long_file, index=False, float_format="%.6f")
wide_file = OUTPUT_FOLDER / "firm_relevance_scores_wide.csv"
wide.sort_values(["sample", "gvkey"]).to_csv(wide_file, index=False, float_format="%.6f")
correlations_file = OUTPUT_FOLDER / "measure_correlations.csv"
correlations.round(4).to_csv(correlations_file)
print(f"Saved: {long_file} ({len(scores_out):,} rows)")
print(f"Saved: {wide_file} ({len(wide):,} rows, {len(measure_columns)} measure columns)")
print(f"Saved: {correlations_file} ({CORRELATION_SAMPLE} sample, Spearman)")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.3/firm_relevance_scores.csv (18,990 rows)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.3/firm_relevance_scores_wide.csv (1,899 rows, 10 measure columns)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.3/measure_correlations.csv (all_years sample, Spearman)
